<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/GAN-WK-11.A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Install required libraries
!pip install -q transformers[torch] datasets

import torch
import shutil
import os
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

# 2. Prepare the dataset
fairy_tale_data = [
    "Once upon a time in a magical forest, there lived a tiny dragon who breathed bubbles instead of fire.",
    "The trees whispered secrets to the wind, and the flowers danced under the moonlight.",
    "Every evening, the silver owl would sing a lullaby that turned the leaves into gold.",
    "Hidden deep within the brambles was a crystal spring that granted wishes to those with kind hearts.",
    "The brave knight did not carry a sword, but a flute that could calm the fiercest of storms."
]

raw_dataset = Dataset.from_dict({"text": fairy_tale_data})

# 3. Load Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 4. Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Clean up old directory manually and define Training Arguments
output_dir = "./gpt2-fairy-tale"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir) # This replaces 'overwrite_output_dir'

model = GPT2LMHeadModel.from_pretrained(model_name)
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=20,
    per_device_train_batch_size=1,
    save_strategy="no",
    logging_steps=5,
    report_to="none" # Prevents unnecessary login prompts
)

# 6. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("Starting Fine-tuning...")
trainer.train()

# 7. Conditioned Generation
prompt = "Once upon a time in a magical forest..."
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids,
    max_length=100,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("\n--- GENERATED TEXT ---")
print(generated_text)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.018525
10,2.680051
15,1.917746
20,1.360093
25,0.728896
30,0.686440
35,0.583040
40,0.275931
45,0.209379
50,0.213628


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- GENERATED TEXT ---
Once upon a time in a magical forest...there lived a tiny dragon who breathed bubbles instead of fire.

Sometimes, those bubbles turned the leaves into gold.
